## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted successfully!')

## Cell 2 — Set Dataset Paths

In [ ]:
import os

# ✅ Update this path if your folder name is different
DATASET_PATH = '/content/drive/MyDrive/OncoAi Dataset'
CANCER_PATH = os.path.join(DATASET_PATH, 'CANCER')
NONCANCER_PATH = os.path.join(DATASET_PATH, 'NON CANCER')

print('Dataset path:', DATASET_PATH)
print('CANCER path exists:', os.path.exists(CANCER_PATH))
print('NON CANCER path exists:', os.path.exists(NONCANCER_PATH))

## Cell 3 — Count Images Per Class

In [ ]:
cancer_files = [f for f in os.listdir(CANCER_PATH) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
noncancer_files = [f for f in os.listdir(NONCANCER_PATH) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

print('=' * 40)
print(f'  CANCER images     : {len(cancer_files)}')
print(f'  NON CANCER images : {len(noncancer_files)}')
print(f'  TOTAL             : {len(cancer_files) + len(noncancer_files)}')
print('=' * 40)
print(f'  Class ratio (Cancer:NonCancer) = {len(cancer_files)}:{len(noncancer_files)}')

## Cell 4 — Plot Class Distribution Bar Chart

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

classes = ['CANCER', 'NON CANCER']
counts = [len(cancer_files), len(noncancer_files)]
colors = ['#e74c3c', '#2ecc71']

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(classes, counts, color=colors, width=0.4, edgecolor='black')

for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(count), ha='center', va='bottom', fontsize=14, fontweight='bold')

ax.set_title('ONCOAi Dataset — Class Distribution', fontsize=15, fontweight='bold', pad=15)
ax.set_ylabel('Number of Images', fontsize=12)
ax.set_xlabel('Class', fontsize=12)
ax.set_ylim(0, max(counts) + 80)
ax.grid(axis='y', linestyle='--', alpha=0.5)

# Save to Drive
SAVE_PATH = os.path.join(DATASET_PATH, 'class_distribution.png')
plt.tight_layout()
plt.savefig(SAVE_PATH, dpi=150)
plt.show()
print(f'Chart saved to: {SAVE_PATH}')

## Cell 5 — Check Image Sizes and Formats

In [ ]:
from PIL import Image
import collections

def get_image_stats(folder, files):
    sizes = []
    formats = []
    corrupt = []
    for f in files:
        try:
            img = Image.open(os.path.join(folder, f))
            sizes.append(img.size)  # (width, height)
            formats.append(img.format)
        except Exception:
            corrupt.append(f)
    return sizes, formats, corrupt

print('Checking CANCER images...')
c_sizes, c_formats, c_corrupt = get_image_stats(CANCER_PATH, cancer_files)

print('Checking NON CANCER images...')
nc_sizes, nc_formats, nc_corrupt = get_image_stats(NONCANCER_PATH, noncancer_files)

all_sizes = c_sizes + nc_sizes
unique_sizes = collections.Counter(all_sizes)

print('\n--- Image Size Summary ---')
print(f'Unique sizes found: {len(unique_sizes)}')
print('Top 5 most common sizes:')
for size, count in unique_sizes.most_common(5):
    print(f'  {size[0]}x{size[1]} px  →  {count} images')

print('\n--- Format Summary ---')
all_formats = collections.Counter(c_formats + nc_formats)
for fmt, count in all_formats.items():
    print(f'  {fmt}: {count} images')

print('\n--- Corrupt Files ---')
print(f'CANCER corrupt: {len(c_corrupt)}')
print(f'NON CANCER corrupt: {len(nc_corrupt)}')
if c_corrupt: print('CANCER corrupt files:', c_corrupt)
if nc_corrupt: print('NON CANCER corrupt files:', nc_corrupt)

## Cell 6 — Display Sample Images from Each Class

In [ ]:
import random
import numpy as np

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle('ONCOAi Dataset — Sample Images', fontsize=16, fontweight='bold', y=1.01)

# 5 CANCER samples
cancer_samples = random.sample(cancer_files, 5)
for i, fname in enumerate(cancer_samples):
    img = Image.open(os.path.join(CANCER_PATH, fname))
    axes[0][i].imshow(img)
    axes[0][i].set_title('CANCER', color='red', fontweight='bold')
    axes[0][i].axis('off')

# 5 NON CANCER samples
noncancer_samples = random.sample(noncancer_files, 5)
for i, fname in enumerate(noncancer_samples):
    img = Image.open(os.path.join(NONCANCER_PATH, fname))
    axes[1][i].imshow(img)
    axes[1][i].set_title('NON CANCER', color='green', fontweight='bold')
    axes[1][i].axis('off')

plt.tight_layout()
SAMPLE_SAVE = os.path.join(DATASET_PATH, 'sample_images.png')
plt.savefig(SAMPLE_SAVE, dpi=150, bbox_inches='tight')
plt.show()
print(f'Sample image grid saved to: {SAMPLE_SAVE}')

## Cell 7 — Pixel Intensity Distribution

In [ ]:
def avg_pixel_intensity(folder, files, n=50):
    """Sample n images and return average pixel intensities"""
    intensities = []
    sample = random.sample(files, min(n, len(files)))
    for f in sample:
        img = Image.open(os.path.join(folder, f)).convert('RGB').resize((224, 224))
        arr = np.array(img).flatten()
        intensities.extend(arr.tolist())
    return intensities

cancer_px = avg_pixel_intensity(CANCER_PATH, cancer_files)
noncancer_px = avg_pixel_intensity(NONCANCER_PATH, noncancer_files)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(cancer_px, bins=50, alpha=0.6, color='red', label='CANCER', density=True)
ax.hist(noncancer_px, bins=50, alpha=0.6, color='green', label='NON CANCER', density=True)
ax.set_title('Pixel Intensity Distribution by Class', fontsize=13, fontweight='bold')
ax.set_xlabel('Pixel Value (0-255)')
ax.set_ylabel('Density')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(DATASET_PATH, 'pixel_distribution.png'), dpi=150)
plt.show()
print('Pixel intensity distribution saved!')

## Cell 8 — Summary Report

In [ ]:
print('=' * 50)
print('       ONCOAi DATASET EXPLORATION SUMMARY')
print('=' * 50)
print(f'  Total Images        : {len(cancer_files) + len(noncancer_files)}')
print(f'  CANCER              : {len(cancer_files)}')
print(f'  NON CANCER          : {len(noncancer_files)}')
print(f'  Class Imbalance     : {len(cancer_files)/len(noncancer_files):.1f}:1 (Cancer:NonCancer)')
print(f'  Corrupt Images      : {len(c_corrupt) + len(nc_corrupt)}')
print(f'  Unique Image Sizes  : {len(unique_sizes)}')
print('  Target Size         : 224 x 224 px (for model input)')
print('  Train/Val/Test Split: 70% / 15% / 15%')
print('=' * 50)
print('\n✅ Data Exploration Complete!')
print('Next step → Run 02_preprocessing.ipynb')